# Base de Dados

## Importação

In [130]:
import pandas as pd
import inflection

df = pd.read_csv('../dataset/zomato.csv')
df1 = df.copy()

## Dados do Projeto

In [131]:
# Renomear as colunas do DataFrame
def rename_columns(dataframe):
    df = dataframe.copy()
    title = lambda x: inflection.titleize(x)
    snakecase = lambda x: inflection.underscore(x)
    spaces = lambda x: x.replace(" ", "")
    cols_old = list(df.columns)
    cols_old = list(map(title, cols_old))
    cols_old = list(map(spaces, cols_old))
    cols_new = list(map(snakecase, cols_old))
    df.columns = cols_new
    return df
df1 = rename_columns(df1)

# Substituição dos códigos dos países pelos nomes dos países
COUNTRIES = {
1: "India",
14: "Australia",
30: "Brazil",
37: "Canada",
94: "Indonesia",
148: "New Zeland",
162: "Philippines",
166: "Qatar",
184: "Singapure",
189: "South Africa",
191: "Sri Lanka",
208: "Turkey",
214: "United Arab Emirates",
215: "England",
216: "United States of America",
}
def country_name(country_id):
    return COUNTRIES[country_id]
df1['country_code'] = df1.loc[:, 'country_code'].apply(lambda x: country_name(x))

# Criação do Tipo de Categoria de Comida
def create_price_tye(price_range):
    if price_range == 1:
        return "cheap"
    elif price_range == 2:
        return "normal"
    elif price_range == 3:
        return "expensive"
    else:
        return "gourmet"
df1['price_range'] = df1.loc[:, 'price_range'].apply(lambda x: create_price_tye(x))

# Criação do nome das Cores
COLORS = {
    "3F7E00": "darkgreen",
    "5BA829": "green",
    "9ACD32": "lightgreen",
    "CDD614": "orange",
    "FFBA00": "red",
    "CBCBC8": "darkred",
    "FF7800": "darkred",
}
def color_name(color_code):
    return COLORS[color_code]

# Categorização por tipo de culinária
df1['cuisines'] = df1['cuisines'].astype(str)
df1["cuisines"] = df1.loc[:, "cuisines"].apply(lambda x: x.split(",")[0])

## Visualização dos Dados

In [132]:
colunas = list(df1.columns)
for col in colunas:
    print(f'{col} - {len(df1[col].unique())}')

restaurant_id - 6942
restaurant_name - 5914
country_code - 15
city - 125
address - 6760
locality - 2272
locality_verbose - 2357
longitude - 6846
latitude - 6833
cuisines - 166
average_cost_for_two - 156
currency - 12
has_table_booking - 2
has_online_delivery - 2
is_delivering_now - 2
switch_to_order_menu - 1
price_range - 4
aggregate_rating - 30
rating_color - 7
rating_text - 28
votes - 1739


In [133]:
na = ['nan']
colunas = list(df1.columns)
for col in colunas:
    print(f'{col} tem {len(df1.loc[df1[col].isin(na),:])} {na[0]}')

restaurant_id tem 0 nan
restaurant_name tem 0 nan
country_code tem 0 nan
city tem 0 nan
address tem 0 nan
locality tem 0 nan
locality_verbose tem 0 nan
longitude tem 0 nan
latitude tem 0 nan
cuisines tem 15 nan
average_cost_for_two tem 0 nan
currency tem 0 nan
has_table_booking tem 0 nan
has_online_delivery tem 0 nan
is_delivering_now tem 0 nan
switch_to_order_menu tem 0 nan
price_range tem 0 nan
aggregate_rating tem 0 nan
rating_color tem 0 nan
rating_text tem 0 nan
votes tem 0 nan


In [134]:
df_aux = df1['cuisines'].value_counts().sort_values().reset_index()
cities = list(df_aux.loc[df_aux['count'] == 1, 'cuisines'])
df_aux2 = df1[df1['cuisines'].isin(cities)]
cities2 = list(df_aux2.loc[(df_aux2['votes'] < 100), 'cuisines'])
df_aux3 = df1[df1['cuisines'].isin(cities2)]
df_aux3.loc[(df_aux3['aggregate_rating'] == 0.0), 'cuisines']

273         Mineira
7011    Drinks Only
Name: cuisines, dtype: object

In [135]:
len(df1) == df1['restaurant_id'].nunique()

False

## Limpeza de Dados

In [136]:
# Limpeza dos dados
df1 = df1.drop(labels=['switch_to_order_menu'], axis='columns')

df1 = df1.loc[(df1['cuisines'] != 'nan'),:].copy()
df1 = df1.loc[(df1['cuisines'] != 'Drinks Only'),:].copy()
df1 = df1.loc[(df1['cuisines'] != 'Mineira'),:].copy()
df1 = df1.drop_duplicates().reset_index(drop=True)

In [137]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6927 entries, 0 to 6926
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   restaurant_id         6927 non-null   int64  
 1   restaurant_name       6927 non-null   object 
 2   country_code          6927 non-null   object 
 3   city                  6927 non-null   object 
 4   address               6927 non-null   object 
 5   locality              6927 non-null   object 
 6   locality_verbose      6927 non-null   object 
 7   longitude             6927 non-null   float64
 8   latitude              6927 non-null   float64
 9   cuisines              6927 non-null   object 
 10  average_cost_for_two  6927 non-null   int64  
 11  currency              6927 non-null   object 
 12  has_table_booking     6927 non-null   int64  
 13  has_online_delivery   6927 non-null   int64  
 14  is_delivering_now     6927 non-null   int64  
 15  price_range          

# Perguntas

## Geral

### 1. Quantos restaurantes únicos estão registrados?

In [84]:
df1.loc[:, 'restaurant_id'].nunique()

6927

### 2. Quantos países únicos estão registrados?

In [85]:
df1.loc[:, 'country_code'].nunique()

15

### 3. Quantas cidades únicas estão registradas?

In [86]:
df1.loc[:, 'city'].nunique()

125

### 4. Qual o total de avaliações feitas?

In [87]:
df1.loc[:, 'votes'].sum()

np.int64(4194530)

### 5. Qual o total de tipos de culinária registrados?

In [117]:
df1.loc[:, 'cuisines'].nunique()

163

## Pais


### 1. Qual o nome do país que possui mais cidades registradas?

In [11]:
df_aux = df1.loc[:, ['country_code', 'city']].groupby('country_code').nunique().sort_values(by='city', ascending=False).reset_index()
df_aux.iloc[0,0]

'India'

### 2. Qual o nome do país que possui mais restaurantes registrados?

In [119]:
df_aux = df1.loc[:, ['country_code', 'restaurant_id']].groupby('country_code').nunique().sort_values(by='restaurant_id', ascending=False).reset_index()
df_aux.iloc[0,0]

'India'

### 3. Qual o nome do país que possui mais restaurantes com o nível de preço igual a 4 registrados?

In [120]:
df_aux = df1.loc[df1['price_range'] == 'gourmet', ['country_code', 'restaurant_id']].groupby('country_code').nunique().sort_values(by='restaurant_id', ascending=False).reset_index()
df_aux.iloc[0,0]

'United States of America'

### 4. Qual o nome do país que possui a maior quantidade de tipos de culinária distintos?

In [121]:
df_aux = df1.loc[:, ['country_code', 'cuisines']].groupby('country_code').nunique().sort_values(by='cuisines', ascending=False).reset_index()
df_aux.iloc[0,0]

'India'

### 5. Qual o nome do país que possui a maior quantidade de avaliações feitas?

In [122]:
df_aux = df1.loc[:, ['country_code', 'votes']].groupby('country_code').sum().sort_values(by='votes', ascending=False).reset_index()
df_aux.iloc[0,0]

'India'

### 6. Qual o nome do país que possui a maior quantidade de restaurantes que fazem entrega?

In [ ]:
df_aux = df1.loc[df1['has_online_delivery'] == 1]
df_aux.loc[:, ['country_code', 'restaurant_id']].groupby('country_code').count().sort_values(by='restaurant_id', ascending=False).reset_index()

,country_code,restaurant_id
0,India,2177
1,United Arab Emirates,205
2,Qatar,37
3,Philippines,9


In [ ]:
df_aux = df1.loc[(df1['has_online_delivery'] == 1), ['country_code', 'restaurant_id']].groupby('country_code').count().sort_values(by='restaurant_id', ascending=False).reset_index()
df_aux.iloc[0,0]

'India'

In [159]:
df1.loc[(df1['has_online_delivery'] == 1), ['country_code', 'restaurant_id']].groupby('country_code').count().sort_values(by='restaurant_id', ascending=False).reset_index().iloc[0,0]

'India'

### 7. Qual o nome do país que possui a maior quantidade de restaurantes que aceitam reservas?

In [161]:
df1.loc[(df1['has_table_booking'] == 1), ['country_code', 'restaurant_id']].groupby('country_code').count().sort_values(by='restaurant_id', ascending=False).reset_index().iloc[0,0]

'India'

### 8. Qual o nome do país que possui, na média, a maior quantidade de avaliações registrada?

In [167]:
df1.loc[:, ['country_code', 'votes']].groupby('country_code').mean().sort_values(by='votes', ascending=False).reset_index().iloc[0,0]

'Indonesia'

### 9. Qual o nome do país que possui, na média, a maior nota média registrada?

In [169]:
df1.loc[:, ['country_code', 'aggregate_rating']].groupby('country_code').mean().sort_values(by='aggregate_rating', ascending=False).reset_index().iloc[0,0]

'Indonesia'

### 10. Qual o nome do país que possui, na média, a menor nota média registradas?

In [170]:
df1.loc[:, ['country_code', 'aggregate_rating']].groupby('country_code').mean().sort_values(by='aggregate_rating', ascending=True).reset_index().iloc[0,0]

'Brazil'

### 11. Qual a média de preço de um prato para dois por país?

In [177]:

df1.loc[:, ['country_code', 'average_cost_for_two']].groupby('country_code').mean().sort_values(by='average_cost_for_two', ascending=False).reset_index().iloc[0,0]

'Indonesia'

## Cidade

### 1. Qual o nome da cidade que possui mais restaurantes registrados?

In [127]:
df1.loc[:, ['city', 'restaurant_id']].groupby('city').nunique().sort_values(by='restaurant_id', ascending=False).reset_index().iloc[0,0]

'Abu Dhabi'

In [ ]:
df_aux = df1.loc[df1['cuisines'] == cuisine, ['restaurant_name', 'restaurant_id', 'aggregate_rating']].groupby(['restaurant_name', 'restaurant_id']).mean().sort_values(by='aggregate_rating', ascending=True).reset_index()
restaurante_maior = df_aux.loc[df_aux['aggregate_rating'] == df_aux['aggregate_rating'].max(), ['restaurant_name', 'restaurant_id']].sort_values(by='restaurant_id', ascending=True).iloc[0,0]

In [128]:
df1.loc[:, ['country_code','city', 'restaurant_id']].groupby(['city', 'country_code']).nunique().sort_values(by='restaurant_id', ascending=False).reset_index()

,city,country_code,restaurant_id
0,Abu Dhabi,United Arab Emirates,80
1,Mangalore,India,80
2,Edinburgh,England,80
3,Glasgow,England,80
4,Goa,India,80
...,...,...,...
120,San Juan City,Philippines,1
121,Roodepoort,South Africa,1
122,Muntinlupa City,Philippines,1
123,Johannesburg South,South Africa,1


### 2. Qual o nome da cidade que possui mais restaurantes com nota média acima de 4?

In [18]:
df1.loc[(df1['aggregate_rating'] > 4), ['city', 'restaurant_id']].groupby('city').nunique().sort_values(by='restaurant_id', ascending=False).reset_index().iloc[0,0]

'Bangalore'

### 3. Qual o nome da cidade que possui mais restaurantes com nota média abaixo de 2.5?

In [19]:
df1.loc[(df1['aggregate_rating'] < 2.5), ['city', 'restaurant_id']].groupby('city').nunique().sort_values(by='restaurant_id', ascending=False).reset_index().iloc[0,0]

'Gangtok'

### 4. Qual o nome da cidade que possui o maior valor médio de um prato para dois?

In [22]:
df1.loc[:, ['city', 'average_cost_for_two']].groupby('city').mean().sort_values(by='average_cost_for_two', ascending=False).reset_index().iloc[0,0]

'Adelaide'

### 5. Qual o nome da cidade que possui a maior quantidade de tipos de culinária distintas?

In [23]:
df1.loc[:, ['city', 'cuisines']].groupby('city').nunique().sort_values(by='cuisines', ascending=False).reset_index().iloc[0,0]

'Birmingham'

### 6. Qual o nome da cidade que possui a maior quantidade de restaurantes que fazem reservas?

In [25]:
df1.loc[(df1['has_table_booking'] == 1), ['city', 'restaurant_id']].groupby('city').count().sort_values(by='restaurant_id', ascending=False).reset_index().iloc[0,0]

'Bangalore'

### 7. Qual o nome da cidade que possui a maior quantidade de restaurantes que fazem entregas?

In [81]:
df_aux = df1.loc[(df1['is_delivering_now'] == 1), ['city', 'restaurant_id']].groupby('city').count().sort_values(by='restaurant_id', ascending=False).reset_index()
cities = list(df_aux.loc[(df_aux['restaurant_id'] == df_aux['restaurant_id'].max()),'city'])
df_aux2 = df1[df1['city'].isin(cities)]
df_aux2.loc[:, ['city', 'restaurant_id']].sort_values(by='restaurant_id', ascending=True).reset_index().iloc[0,1]

'Amritsar'

### 8. Qual o nome da cidade que possui a maior quantidade de restaurantes que aceitam pedidos online?

In [83]:
df1.loc[(df1['has_online_delivery'] == 1), ['city', 'restaurant_id']].groupby('city').count().sort_values(by='restaurant_id', ascending=False).reset_index().iloc[0,0]

'Bhopal'

## Restaurantes

### 1. Qual o nome do restaurante que possui a maior quantidade de avaliações?

In [20]:
df1.loc[:,['restaurant_name', 'votes']].groupby('restaurant_name').sum().sort_values(by='votes', ascending=False).reset_index().iloc[0,0]

"Domino's Pizza"

### 2. Qual o nome do restaurante com a maior nota média?

In [31]:
df_aux = df1.loc[:,['restaurant_name','restaurant_id', 'aggregate_rating']].groupby(['restaurant_name','restaurant_id']).mean().sort_values(by='aggregate_rating', ascending=False).reset_index()
df_aux.loc[df_aux['aggregate_rating'] == df_aux['aggregate_rating'].max(), ['restaurant_name','restaurant_id']].groupby('restaurant_name').min().sort_values(by='restaurant_id', ascending=True).reset_index().iloc[0,0]

'Indian Grill Room'

### 3. Qual o nome do restaurante que possui o maior valor de uma prato para duas pessoas?

In [37]:
df1.loc[:, ['restaurant_name', 'average_cost_for_two']].groupby('restaurant_name').max().sort_values(by='average_cost_for_two', ascending=False).reset_index().iloc[0,0]

"d'Arry's Verandah Restaurant"

### 4. Qual o nome do restaurante de tipo de culinária brasileira que possui a menor média de avaliação?

In [52]:
df_aux = df1.loc[df1['cuisines'] == 'Brazilian', ['restaurant_name', 'cuisines','restaurant_id', 'aggregate_rating']].groupby(['restaurant_name', 'cuisines','restaurant_id']).min().sort_values(by='aggregate_rating', ascending=True).reset_index()
df_aux.loc[df_aux['aggregate_rating'] == df_aux['aggregate_rating'].min(), ['restaurant_name','restaurant_id']].groupby('restaurant_name').min().sort_values(by='restaurant_id', ascending=True).reset_index().iloc[0,0]

'Loca Como tu Madre'

### 5. Qual o nome do restaurante de tipo de culinária brasileira, e que é do Brasil, que possui a maior média de avaliação?

In [56]:
df_aux = df1.loc[(df1['cuisines'] == 'Brazilian') & (df1['country_code'] == 'Brazil'), ['restaurant_name', 'cuisines','restaurant_id', 'aggregate_rating']].groupby(['restaurant_name', 'cuisines','restaurant_id']).max().sort_values(by='aggregate_rating', ascending=False).reset_index()
df_aux.loc[df_aux['aggregate_rating'] == df_aux['aggregate_rating'].max(), ['restaurant_name','restaurant_id']].groupby('restaurant_name').min().sort_values(by='restaurant_id', ascending=True).reset_index().iloc[0,0]

'Braseiro da Gávea'

### 6. Os restaurantes que aceitam pedido online são também, na média, os restaurantes que mais possuem avaliações registradas?

In [29]:
avg_votes = df1['votes'].mean().round(2)
avg_votes_online_delivery = df1.loc[df1['has_online_delivery'] == 1, 'votes'].mean().round(2)

top_restaurants_online_delivery = df1.loc[df1['has_online_delivery'] == 1, ['restaurant_name', 'votes']].groupby('restaurant_name').mean().sort_values(by='votes', ascending=False).reset_index().round(2).head(10)

print('Top 10 restaurantes com entrega online e suas médias de votos:')
print(top_restaurants_online_delivery)

print(f'Média de votos: {avg_votes}')
print(f'Média de votos para restaurantes com entrega online: {avg_votes_online_delivery}')

if avg_votes_online_delivery > avg_votes:
    print('Restaurantes com entrega online têm mais votos, em média, do que os restaurantes sem entrega online.')
elif avg_votes_online_delivery < avg_votes:
    print('Restaurantes sem entrega online têm mais votos, em média, do que os restaurantes com entrega online.')
else:
    print('Restaurantes com entrega online e sem entrega online têm a mesma média de votos.')

Top 10 restaurantes com entrega online e suas médias de votos:
                  restaurant_name     votes
0                        Bawarchi  13903.67
1                Hauz Khas Social  13627.00
2  Shah Ghouse Hotel & Restaurant  11836.00
3     Byg Brewski Brewing Company  11543.00
4                       Peter Cat  11476.00
5                    Captain Cook   9076.00
6                   Colaba Social   8369.00
7                    Prithvi Cafe   7995.00
8                           BarBQ   7744.00
9                Lucky Restaurant   7565.00
Média de votos: 605.53
Média de votos para restaurantes com entrega online: 838.82
Restaurantes com entrega online têm mais votos, em média, do que os restaurantes sem entrega online.


### 7. Os restaurantes que fazem reservas são também, na média, os restaurantes que possuem o maior valor médio de um prato para duas pessoas?

In [33]:
avg_ft = df1['average_cost_for_two'].mean().round(2)
avg_ft_table_booking = df1.loc[df1['has_table_booking'] == 1, 'average_cost_for_two'].mean().round(2)

top_restaurants_table_booking = df1.loc[df1['has_table_booking'] == 1, ['restaurant_name', 'average_cost_for_two']].groupby('restaurant_name').mean().sort_values(by='average_cost_for_two', ascending=False).reset_index().round(2).head(10)

print('Top 10 restaurantes com reserva em mesa e suas médias de custo para dois:')
print(top_restaurants_table_booking)

print(f'Média de custo para dois: {avg_ft}')
print(f'Média de custo para dois para restaurantes com reserva em mesa: {avg_ft_table_booking}')

if avg_ft_table_booking > avg_ft:
    print('Restaurantes com reserva em mesa têm mais custo para dois, em média, do que os restaurantes sem reserva em mesa.')
elif avg_ft_table_booking < avg_ft:
    print('Restaurantes sem reserva em mesa têm mais custo para dois, em média, do que os restaurantes com reserva em mesa.')
else:
    print('Restaurantes com reserva em mesa e sem reserva em mesa têm a mesma média de custo para dois.')

Top 10 restaurantes com reserva em mesa e suas médias de custo para dois:
                restaurant_name  average_cost_for_two
0  d'Arry's Verandah Restaurant            25000017.0
1                3 Wise Monkeys              450000.0
2                    Tapas Club              400000.0
3              Hokkaido Izakaya              350000.0
4             NOBLE by Zab Thai              300000.0
5      WAKI Japanese BBQ Dining              300000.0
6         Magal Mapogalmegi BBQ              300000.0
7          The Cutt Grill House              300000.0
8                  Itacho Sushi              250000.0
9                SGD Tofu House              250000.0
Média de custo para dois: 7522.21
Média de custo para dois para restaurantes com reserva em mesa: 69998.42
Restaurantes com reserva em mesa têm mais custo para dois, em média, do que os restaurantes sem reserva em mesa.


### 8. Os restaurantes do tipo de culinária japonesa dos Estados Unidos da América possuem um valor médio de prato para duas pessoas maior que as churrascarias americanas (BBQ)?

In [35]:
avg_japa = df1.loc[(df1['cuisines'] == 'Japanese') & (df1['country_code'] == 'United States of America'),'average_cost_for_two'].mean()
avg_bbq = df1.loc[(df1['cuisines'] == 'BBQ') & (df1['country_code'] == 'United States of America'),'average_cost_for_two'].mean()

print(f'Média de custo para dois para restaurantes japoneses nos Estados Unidos: {avg_japa:.2f}')
print(f'Média de custo para dois para restaurantes de churrasco nos Estados Unidos: {avg_bbq:.2f}')

if avg_japa > avg_bbq:
    print('Restaurantes japoneses nos Estados Unidos têm mais custo para dois, em média, do que os restaurantes de churrasco nos Estados Unidos.')
elif avg_japa < avg_bbq:
    print('Restaurantes de churrasco nos Estados Unidos têm mais custo para dois, em média, do que os restaurantes japoneses nos Estados Unidos.')

Média de custo para dois para restaurantes japoneses nos Estados Unidos: 56.41
Média de custo para dois para restaurantes de churrasco nos Estados Unidos: 39.64
Restaurantes japoneses nos Estados Unidos têm mais custo para dois, em média, do que os restaurantes de churrasco nos Estados Unidos.


## Tipos de Culinárias

### 1. Dos restaurantes que possuem o tipo de culinária italiana, qual o nome do restaurante com a maior média de avaliação?

In [141]:
df_aux = df1.loc[df1['cuisines'] == 'Italian', ['restaurant_name', 'restaurant_id', 'aggregate_rating']].groupby(['restaurant_name', 'restaurant_id']).mean().sort_values(by='aggregate_rating', ascending=True).reset_index()
df_aux = df_aux.loc[df_aux['aggregate_rating'] == df_aux['aggregate_rating'].max(), ['restaurant_name', 'restaurant_id']].sort_values(by='restaurant_id', ascending=True)

df_aux = df1.loc[df1['restaurant_id'] == df_aux.iloc[0,1],['restaurant_name','country_code','city','average_cost_for_two','currency','cuisines','aggregate_rating']]
df_aux

,restaurant_name,country_code,city,average_cost_for_two,currency,cuisines,aggregate_rating
4994,Darshan,India,Pune,700,Indian Rupees(Rs.),Italian,4.9


In [86]:
def cuisines_votes_restaurant(media, cuisine):
    if media == 'maior':
        df_aux = df1.loc[df1['cuisines'] == cuisine, ['restaurant_name', 'restaurant_id', 'aggregate_rating']].groupby(['restaurant_name', 'restaurant_id']).mean().sort_values(by='aggregate_rating', ascending=True).reset_index()
        restaurante_maior = df_aux.loc[df_aux['aggregate_rating'] == df_aux['aggregate_rating'].max(), ['restaurant_name', 'restaurant_id']].sort_values(by='restaurant_id', ascending=True).iloc[0,0]
        return print(f'O restaurante com a maior média de votos para a culinária {cuisine} é: {restaurante_maior}')
    elif media == 'menor':
        df_aux = df1.loc[df1['cuisines'] == cuisine, ['restaurant_name', 'restaurant_id', 'aggregate_rating']].groupby(['restaurant_name', 'restaurant_id']).mean().sort_values(by='aggregate_rating', ascending=True).reset_index()
        restaurante_menor = df_aux.loc[df_aux['aggregate_rating'] == df_aux['aggregate_rating'].min(), ['restaurant_name', 'restaurant_id']].sort_values(by='restaurant_id', ascending=True).iloc[0,0]
        return print(f'O restaurante com a menor média de votos para a culinária {cuisine} é: {restaurante_menor}')
    else:
        return print('Opção de média inválida. Por favor, escolha "maior" ou "menor".')

cuisines_votes_restaurant('maior', 'Italian')

O restaurante com a maior média de votos para a culinária Italian é: Darshan


### 2. Dos restaurantes que possuem o tipo de culinária italiana, qual o nome do restaurante com a menor média de avaliação?

In [89]:
cuisines_votes_restaurant('menor', 'Italian')

O restaurante com a menor média de votos para a culinária Italian é: Avenida Paulista


### 3. Dos restaurantes que possuem o tipo de culinária americana, qual o nome do restaurante com a maior média de avaliação?

In [88]:
cuisines_votes_restaurant('maior', 'American')

O restaurante com a maior média de votos para a culinária American é: Burger & Lobster


### 4. Dos restaurantes que possuem o tipo de culinária americana, qual o nome do restaurante com a menor média de avaliação?

In [90]:
cuisines_votes_restaurant('menor', 'American')

O restaurante com a menor média de votos para a culinária American é: Alston Bar & Beef


### 5. Dos restaurantes que possuem o tipo de culinária árabe, qual o nome do restaurante com a maior média de avaliação?

In [91]:
cuisines_votes_restaurant('maior', 'Arabian')

O restaurante com a maior média de votos para a culinária Arabian é: Mandi@36


### 6. Dos restaurantes que possuem o tipo de culinária árabe, qual o nome do restaurante com a menor média de avaliação?

In [92]:
cuisines_votes_restaurant('menor', 'Arabian')

O restaurante com a menor média de votos para a culinária Arabian é: Raful


### 7. Dos restaurantes que possuem o tipo de culinária japonesa, qual o nome do restaurante com a maior média de avaliação?

In [94]:
cuisines_votes_restaurant('maior', 'Japanese')

O restaurante com a maior média de votos para a culinária Japanese é: Sushi Samba


### 8. Dos restaurantes que possuem o tipo de culinária japonesa, qual o nome do restaurante com a menor média de avaliação?

In [95]:
cuisines_votes_restaurant('menor', 'Japanese')

O restaurante com a menor média de votos para a culinária Japanese é: Banzai Sushi


### 9. Dos restaurantes que possuem o tipo de culinária caseira, qual o nome do restaurante com a maior média de avaliação?

In [104]:
cuisines_votes_restaurant('maior', 'Home-made')

O restaurante com a maior média de votos para a culinária Home-made é: Kanaat Lokantası


### 10. Dos restaurantes que possuem o tipo de culinária caseira, qual o nome do restaurante com a menor média de avaliação?

In [105]:
cuisines_votes_restaurant('menor', 'Home-made')

O restaurante com a menor média de votos para a culinária Home-made é: GurMekan Restaurant


### 11. Qual o tipo de culinária que possui o maior valor médio de um prato para duas pessoas?

In [110]:
df1.loc[:, ['cuisines', 'average_cost_for_two']].groupby('cuisines').mean().sort_values(by='average_cost_for_two', ascending=False).reset_index().iloc[0,0]

'Modern Australian'

### 12. Qual o tipo de culinária que possui a maior nota média?

In [107]:
df1.loc[:, ['cuisines', 'aggregate_rating']].groupby('cuisines').mean().sort_values(by='aggregate_rating', ascending=False).reset_index().iloc[0,0]

'Others'

### 13. Qual o tipo de culinária que possui mais restaurantes que aceitam pedidos online e fazem entregas?

In [121]:
df1.loc[(df1['has_online_delivery'] == 1) & (df1['has_online_delivery'] == 1), ['cuisines', 'restaurant_id']].groupby('cuisines').count().sort_values(by='restaurant_id', ascending=False).reset_index().iloc[0,0]

'North Indian'